In [ ]:
import cv2
import numpy as np
import onnxruntime
from gpiozero import InputDevice
import time

############################################
# CONFIG
############################################
RADAR_PIN = 17           # OUT pin connected to GPIO17
MODEL_PATH = "best.onnx" # YOLO ONNX model
CLASS_NAMES = ["person", "gun", "knife"]
IMG_SIZE = 640
CONF_THRESH = 0.35
############################################

# Initialize Radar
radar = InputDevice(RADAR_PIN)
print("Radar Ready...")

# Initialize YOLO ONNX Runtime
session = onnxruntime.InferenceSession(MODEL_PATH)
input_name = session.get_inputs()[0].name

# Initialize both cameras
cam0 = cv2.VideoCapture(0)    # USB camera
cam1 = cv2.VideoCapture(1)    # CSI camera (v4l)
print("Cameras Initialized...")

def preprocess(frame):
    img = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))
    img = np.expand_dims(img, axis=0)
    return img

def postprocess(pred, frame_shape):
    results = []
    h, w, _ = frame_shape
    for det in pred:
        x1, y1, x2, y2, conf, cls = det
        if conf < CONF_THRESH:
            continue
        results.append([int(x1), int(y1), int(x2), int(y2), conf, int(cls)])
    return results

def draw_boxes(frame, detections):
    for (x1, y1, x2, y2, conf, cls) in detections:
        label = f"{CLASS_NAMES[cls]} {conf:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, label, (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
    return frame

def run_yolo(frame):
    img = preprocess(frame)
    preds = session.run(None, {input_name: img})
    detections = postprocess(preds[0], frame.shape)
    return detections

############################################
# MAIN LOOP
############################################
while True:

    # Read radar
    motion = radar.is_active

    if not motion:
        print("Radar: NO MOTION → YOLO SLEEP")
        time.sleep(0.3)
        continue

    print("\n⚠️ Motion Detected → YOLO ACTIVATED\n")

    # Capture frames
    ret0, frame0 = cam0.read()
    ret1, frame1 = cam1.read()

    # Run YOLO on both cameras
    if ret0:
        det0 = run_yolo(frame0)
        frame0 = draw_boxes(frame0, det0)

    if ret1:
        det1 = run_yolo(frame1)
        frame1 = draw_boxes(frame1, det1)

    # Display both camera feeds side-by-side
    if ret0 and ret1:
        combined = np.hstack((frame0, frame1))
    elif ret0:
        combined = frame0
    else:
        combined = frame1

    cv2.imshow("Radar + YOLO Fusion", combined)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam0.release()
cam1.release()
cv2.destroyAllWindows()
